In [14]:
import confnotebook

In [15]:
from pathlib import Path

source = Path("../examples/test/bag_date/")

files = sorted(source.glob("*.pdf"))

for i, file in enumerate(files):
    print(f"[{i}] {file.stem}")

[0] 127113
[1] 127116
[2] Сверка БайкАкв от 03.06.26
[3] Ситилинк


In [16]:
IDX_FILE = 1

In [17]:
from vision_core.debug_image_observer import DebugImageObserver

file = files[IDX_FILE]
output_dir = f"../examples/output/{file.stem}"

debug_image_observer = DebugImageObserver(output_dir=output_dir)

In [18]:
from vision_core.pipelines.build_document import DocumentBuildPipeline

pipeline = DocumentBuildPipeline(debug_image=debug_image_observer)

document = pipeline.build(file.read_bytes())

Creating model: ('PP-OCRv5_server_det', 'D:\\projects\\rusal_recon_srv\\repo\\recon_vision\\models\\PP-OCRv5_server_det')
The specified device (GPU) is not available! Switching to CPU instead.
Creating model: ('cyrillic_PP-OCRv5_mobile_rec', 'D:\\projects\\rusal_recon_srv\\repo\\recon_vision\\models\\cyrillic_PP-OCRv5_mobile_rec')
The specified device (GPU) is not available! Switching to CPU instead.
2026-06-16 13:16:23.916 | INFO     | vision_core.pipelines.build_document:build:106 - Обработка страницы 0 с dpi 200...
2026-06-16 13:16:24.049 | INFO     | vision_core.pipelines.build_document:_process_page:190 - Коррекция ориентации и наклона...
2026-06-16 13:16:24.111 | DEBUG    | vision_core.preprocessor.image_orientation:process:47 - Ориентация страницы: 0° с точностью 0.9201
2026-06-16 13:16:24.181 | DEBUG    | vision_core.preprocessor.image_orientation:compute_deskew_angle:119 - Углы наклона страницы: 0.1429°
2026-06-16 13:16:24.373 | DEBUG    | vision_core.debug_image_observer:on_d

In [19]:
from app.infrastructure.services.structured_data_extractor import ReconciliationActExtractor

extractor = ReconciliationActExtractor()
data = await extractor.extract(document)

2026-06-16 13:16:46.685 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_text:70 - summary_text: 778 символов из 1 страниц
2026-06-16 13:16:46.686 | DEBUG    | app.infrastructure.services.extractor.company_ext:_build_summary_cell_texts:92 - summary_cell_texts: ['ПО ДАННЫМ ПРОДАВЦА\nООО "ЛУКОЙЛ-АСТРАХАНЬЭНЕРГО"\nИНН 3016059510', 'ПО ДАННЫМ ПОКУПАТЕЛЯ\nАО "РУСАЛ САЯНОГОРСКИЙ\nАЛЮМИНИЕВЫЙ ЗАВОД\nИНН 1902014500']
2026-06-16 13:16:46.687 | INFO     | app.infrastructure.services.extractor.company_ext:extract_companies:248 - кандидаты: ['ЛУКОЙЛ-АСТРАХАНЬЭНЕРГО, ООО']
2026-06-16 13:16:46.689 | DEBUG    | app.infrastructure.services.extractor.company_ext:_assign_roles:185 - события: ['ОТ ПРОДАВЦА', 'ОТ ПОКУПАТЕЛЯ']
2026-06-16 13:16:46.690 | DEBUG    | app.infrastructure.services.extractor.company_ext:_assign_roles:187 - рабочая пара: ['ЛУКОЙЛ-АСТРАХАНЬЭНЕРГО, ООО']
2026-06-16 13:16:46.691 | DEBUG    | app.infrastructure.services.extractor.company_ext:_assign_roles:2

In [20]:
print(data.debit)

[LedgerEntry(record='САЛЬДО НА 01.01.2026 Г.\nПО ОПЛАТЕ МОЩНОСТИ, В Т.Ч. НДС\nСВЕРНУТОЕ', value=17533.03, date='01.01.2026', row_reference=RowReference(id_table='0', id_row='2', id_col=1, buyer_col=3)), LedgerEntry(record='РАЗВЕРНУТОЕ', value=17533.03, date=None, row_reference=RowReference(id_table='0', id_row='3', id_col=1, buyer_col=3)), LedgerEntry(record='ПО ОПЛАТЕ НЕУСТОЙКИ (ШТРАФОВ, ПЕНИ)', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='4', id_col=1, buyer_col=3)), LedgerEntry(record='ПРИОБРЕТЕНО МОЩНОСТИ\nНА СУММУ,\nВ Т.Ч. НДС', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='5', id_col=1, buyer_col=3)), LedgerEntry(record='НАЧИСЛЕНА НЕУСТОЙКА (ШТРАФЫ, ПЕНИ )', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='6', id_col=1, buyer_col=3)), LedgerEntry(record='ОПЛАЧЕНО:\nМОЩНОСТЬ,\nІВ Т.Ч. НДС', value=0.0, date=None, row_reference=RowReference(id_table='0', id_row='7', id_col=1, buyer_col=3)), LedgerEntry(record='

In [21]:
from app.application.dto.fill_reconciliation_act import FillReconciliationActCommand
from app.domain.entities.process import ProcessState
from app.infrastructure.services.pdf_filler import DocumentPdfFiller

process_state = ProcessState(
    process_id="notebook-test",
    source_pdf=files[IDX_FILE].read_bytes(),
    document_payload=document,
)

comments = """
            По данным АО "РУСАЛ Новокузнецк" на 30.09.2023
            задолженность в пользу АО "РУСАЛ Новокузнецк"
            составляет 13 755 023,24 руб.
            С разногласиями, протокол разногласий прилагается.
            Акт сверки проверен ОУФО ОЦО, ООО "РЦУ".
            Исполнитель: Воробьева Оксана Евгеньевна
            Дата:29.01.2025
            """

# используем значения продавца для заполнения колонок покупателя
command = FillReconciliationActCommand(
    process_id="notebook-test",
    comments=comments,
    debit=data.debit,
    credit=data.credit,
)

filler = DocumentPdfFiller()
filled_pdf = await filler.fill(process_state, command)

2026-06-16 13:16:46.724 | INFO     | app.infrastructure.services.pdf_fill.render:resolve_font_file:221 - найден шрифт: D:\projects\rusal_recon_srv\repo\recon_vision\assets\fonts\LiberationSerif-Regular.ttf
2026-06-16 13:16:46.889 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_aligned_page_images:49 - page=0 dpi=200 source=1656x2339 aligned=1656x2339 canvas=1656x2339
2026-06-16 13:16:46.894 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R2:C3 значение=17533.03
2026-06-16 13:16:46.894 | DEBUG    | app.infrastructure.services.pdf_fill.render:load_font:229 - загружаем шрифт из D:\projects\rusal_recon_srv\repo\recon_vision\assets\fonts\LiberationSerif-Regular.ttf для размера 25
2026-06-16 13:16:46.896 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R3:C3 значение=17533.03
2026-06-16 13:16:46.898 | DEBUG    | app.infrastructure.services.pdf_filler:fill:48 - заполняем таблица=0 R4:C3 значение=0.0
2026-06-16 13

In [22]:
import base64

from IPython.display import HTML, display

b64 = base64.b64encode(filled_pdf).decode()
display(HTML(f'<a href="data:application/pdf;base64,{b64}" download="filled.pdf">Скачать заполненный PDF</a>'))